# Project 2 — วิเคราะห์แคมเปญการตลาดของธนาคาร (Bank Marketing Analysis)

**เป้าหมายของโปรเจกต์:** ฝึกใช้ Python และ SQL เพื่อวิเคราะห์ผลแคมเปญการตลาด และนำผลลัพธ์ไปสรุปเป็นข้อเสนอแนะทางธุรกิจ

**ชุดข้อมูล:** Kaggle Playground Series S5E8 — Binary Classification with a Bank Dataset

ชุดข้อมูลนี้เป็นข้อมูลสังเคราะห์ที่สร้างขึ้นโดยอ้างอิงจากชุดข้อมูล Bank Marketing ต้นฉบับ ไม่ใช่บันทึกธุรกรรมหรือแคมเปญที่เกิดขึ้นจริงในปี 2025

## วิธีใช้งาน
1. ดาวน์โหลดไฟล์ `train.csv` จากแท็บ Data ของ [การแข่งขันบน Kaggle](https://www.kaggle.com/competitions/playground-series-s5e8/data)
2. รันแต่ละ Cell จากบนลงล่าง
3. เมื่อมีหน้าต่างให้อัปโหลดไฟล์ ให้เลือก `train.csv`
4. ตรวจสอบผลการตรวจคุณภาพข้อมูลและผล SQL ก่อนสรุป Insight

**ข้อควรระวัง:** การเปรียบเทียบเชิงพรรณนาไม่ได้พิสูจน์ความเป็นเหตุเป็นผล ตัวแปรเป้าหมาย `y` ใช้ระบุผลการสมัครเงินฝากประจำ

## 1. โหลดชุดข้อมูล
รัน Cell ถัดไป แล้วเลือกอัปโหลดไฟล์ `train.csv` ที่ดาวน์โหลดจาก Kaggle

In [ ]:
from google.colab import files
uploaded = files.upload()

import io
import pandas as pd

csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
if not csv_files:
    raise ValueError("ไม่พบไฟล์ CSV กรุณาอัปโหลด train.csv จาก Kaggle")

preferred = [name for name in csv_files if name.lower() == "train.csv"]
file_name = preferred[0] if preferred else csv_files[0]
df = pd.read_csv(io.BytesIO(uploaded[file_name]))

print(f"ไฟล์ที่โหลด: {file_name}")
print(f"จำนวนแถว: {df.shape[0]:,} | จำนวนคอลัมน์: {df.shape[1]}")
display(df.head())


## 2. ทำความเข้าใจข้อมูล
ตรวจสอบชื่อคอลัมน์ ชนิดข้อมูล ค่าว่าง แถวซ้ำ และสัดส่วนของตัวแปรเป้าหมาย

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
df.info()

print("\nMissing values (top 15):")
display(df.isna().sum().sort_values(ascending=False).head(15).to_frame("missing_count"))

print("\nDuplicate rows:", df.duplicated().sum())

if "y" not in df.columns:
    raise KeyError("Expected target column 'y' was not found. Check that you uploaded the competition train.csv.")

print("\nTarget counts:")
display(df["y"].value_counts(dropna=False).to_frame("count"))

print("\nTarget share (%):")
display((df["y"].value_counts(normalize=True, dropna=False) * 100).round(2).to_frame("percent"))


## 3. ตรวจสอบค่าหมวดหมู่ที่อาจหมายถึงข้อมูลไม่ทราบค่า

ในข้อมูลลักษณะนี้อาจมีค่า `unknown` อยู่ในคอลัมน์หมวดหมู่ ซึ่งไม่เหมือนกับค่าว่าง (`NaN`) เสมอไป เราจะตรวจทั้งจำนวนและสัดส่วนก่อนตัดสินใจจัดการ

**แนวทาง:** อย่าแทน `unknown` ด้วยค่าที่พบบ่อยที่สุดโดยอัตโนมัติ เพราะอาจสร้างข้อมูลที่ไม่มีหลักฐานรองรับ ในการวิเคราะห์หลักให้เก็บค่าไว้เป็นหมวดหมู่หนึ่งและรายงานสัดส่วนก่อน หากมีสัดส่วนสูง ให้ตรวจสอบความหมายจากเอกสารชุดข้อมูล และทำการวิเคราะห์ความไวโดยเปรียบเทียบผลทั้งแบบเก็บไว้และตัดออกเฉพาะเมื่อมีเหตุผลชัดเจน

In [ ]:
# ตรวจหาค่า 'unknown' โดยไม่สนใจตัวพิมพ์เล็ก/ใหญ่ และช่องว่างหัวท้าย
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
unknown_rows = []
for col in categorical_cols:
    normalized = df[col].astype("string").str.strip().str.lower()
    count = int(normalized.eq("unknown").fillna(False).sum())
    if count > 0:
        unknown_rows.append({
            "column": col,
            "unknown_count": count,
            "unknown_pct": round(count / len(df) * 100, 2) if len(df) else 0.0,
        })

unknown_report = pd.DataFrame(unknown_rows, columns=["column", "unknown_count", "unknown_pct"])
if unknown_report.empty:
    print("ไม่พบค่า 'unknown' ในคอลัมน์ข้อความ")
else:
    display(unknown_report.sort_values("unknown_pct", ascending=False))

print("จำนวนค่าว่าง (NaN) ในคอลัมน์หมวดหมู่:")
display(df[categorical_cols].isna().sum().sort_values(ascending=False).to_frame("missing_count"))


## 4. วิเคราะห์ข้อมูลเบื้องต้นด้วย Python (EDA)

ขั้นตอนนี้จะสำรวจข้อมูลและเปรียบเทียบอัตราการสมัครในกลุ่มต่าง ๆ โดยโค้ดจะตรวจสอบก่อนว่าคอลัมน์ที่ต้องการมีอยู่ในชุดข้อมูลหรือไม่

การเปรียบเทียบนี้เป็นเพียงความสัมพันธ์ที่พบในข้อมูล ไม่ได้ยืนยันว่าตัวแปรนั้นเป็นสาเหตุให้ลูกค้าสมัคร

In [ ]:
candidate_columns = ["job", "education", "marital", "contact", "month", "campaign", "previous", "poutcome", "age"]
available_columns = [col for col in candidate_columns if col in df.columns]
print("Available candidate columns:", available_columns)

def segment_summary(data, column, target="y"):
    summary = (
        data.groupby(column, dropna=False)[target]
        .agg(contacts="size", subscriptions=lambda s: (s == "yes").sum())
        .reset_index()
    )
    summary["subscription_rate_pct"] = (
        summary["subscriptions"] / summary["contacts"] * 100
    ).round(2)
    return summary.sort_values(["subscription_rate_pct", "contacts"], ascending=[False, False])

for col in ["job", "contact", "month", "poutcome"]:
    if col in df.columns:
        print(f"\nSubscription rate by {col}:")
        display(segment_summary(df, col))


## 5. แสดงผลด้วยกราฟ
กราฟนี้เปรียบเทียบอัตราการสมัคร ไม่ใช่จำนวนลูกค้าดิบ จึงควรพิจารณาปริมาณข้อมูลของแต่ละกลุ่มควบคู่กัน

In [ ]:
import matplotlib.pyplot as plt

if "contact" in df.columns:
    plot_col = "contact"
elif available_columns:
    plot_col = available_columns[0]
else:
    raise ValueError("No candidate segment columns are available.")

plot_data = segment_summary(df, plot_col).sort_values("subscription_rate_pct", ascending=True)
plt.figure(figsize=(9, 5))
plt.barh(plot_data[plot_col].astype(str), plot_data["subscription_rate_pct"])
plt.title(f"Term-deposit subscription rate by {plot_col}")
plt.xlabel("Subscription rate (%)")
plt.ylabel(plot_col)
plt.tight_layout()
plt.show()


## 6. ฝึก SQL ด้วย SQLite
ขั้นตอนนี้นำข้อมูลจาก DataFrame ไปเก็บในตาราง SQLite ชื่อ `bank_marketing` เพื่อให้ฝึกเขียน SQL ได้ภายใน Notebook เดียวกัน

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
df.to_sql("bank_marketing", conn, if_exists="replace", index=False)

def run_sql(query):
    return pd.read_sql_query(query, conn)

display(run_sql("""
SELECT
    COUNT(*) AS total_records,
    SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) AS subscriptions,
    ROUND(100.0 * SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS subscription_rate_pct
FROM bank_marketing;
"""))


### แบบฝึก SQL 1 — เปรียบเทียบกลุ่มอาชีพ
ลองแก้ไขคำสั่ง SQL เช่น เปลี่ยนลำดับการเรียงผลลัพธ์ หรือปรับจำนวนข้อมูลขั้นต่ำของแต่ละกลุ่ม

In [ ]:
display(run_sql("""
SELECT
    job,
    COUNT(*) AS contacts,
    SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) AS subscriptions,
    ROUND(100.0 * SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS subscription_rate_pct
FROM bank_marketing
GROUP BY job
HAVING COUNT(*) >= 100
ORDER BY subscription_rate_pct DESC;
"""))


### แบบฝึก SQL 2 — เปรียบเทียบช่องทางการติดต่อ
พิจารณาทั้งอัตราการสมัครและจำนวนครั้งที่ติดต่อ กลุ่มที่มีอัตราสูงแต่มีข้อมูลน้อยมากอาจไม่น่าเชื่อถือเท่ากลุ่มที่มีข้อมูลมากกว่า

In [ ]:
import matplotlib.pyplot as plt

if "contact" in df.columns:
    plot_col = "contact"
elif available_columns:
    plot_col = available_columns[0]
else:
    raise ValueError("No candidate segment columns are available.")

plot_data = segment_summary(df, plot_col).sort_values("subscription_rate_pct", ascending=True)
plt.figure(figsize=(9, 5))
plt.barh(plot_data[plot_col].astype(str), plot_data["subscription_rate_pct"])
plt.title(f"Term-deposit subscription rate by {plot_col}")
plt.xlabel("Subscription rate (%)")
plt.ylabel(plot_col)
plt.tight_layout()
plt.show()


### แบบฝึก SQL 3 — เปรียบเทียบระดับการศึกษา

เปรียบเทียบจำนวนระเบียน จำนวนการสมัคร และอัตราการสมัครในแต่ละระดับการศึกษา ตรวจสอบขนาดกลุ่มควบคู่กับอัตราเสมอ

In [ ]:
if "education" in df.columns:
    display(run_sql("""
    SELECT
        education,
        COUNT(*) AS records,
        SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) AS subscriptions,
        ROUND(100.0 * SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS subscription_rate_pct
    FROM bank_marketing
    GROUP BY education
    ORDER BY subscription_rate_pct DESC, records DESC;
    """))
else:
    print("ไม่พบคอลัมน์ education ในชุดข้อมูลนี้")


### แบบฝึก SQL 4 — เปรียบเทียบเดือนที่ติดต่อ

ใช้ผลนี้ดูความแตกต่างตามเดือนในชุดข้อมูล แต่ห้ามสรุปว่าเดือนเป็นสาเหตุโดยตรง เพราะอาจมีปัจจัยร่วมอื่น เช่น ช่องทาง กลุ่มลูกค้า หรือจำนวนครั้งที่ติดต่อ

In [ ]:
if "month" in df.columns:
    display(run_sql("""
    SELECT
        month,
        COUNT(*) AS records,
        SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) AS subscriptions,
        ROUND(100.0 * SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS subscription_rate_pct
    FROM bank_marketing
    GROUP BY month
    ORDER BY subscription_rate_pct DESC, records DESC;
    """))
else:
    print("ไม่พบคอลัมน์ month ในชุดข้อมูลนี้")


### แบบฝึก SQL 5 — จำนวนครั้งที่ติดต่อในแคมเปญปัจจุบัน

คอลัมน์ `campaign` ในชุดข้อมูล Bank Marketing โดยทั่วไปหมายถึงจำนวนครั้งที่ติดต่อผู้ติดต่อรายนั้นในแคมเปญปัจจุบัน (รวมการติดต่อครั้งนี้) ตรวจสอบความหมายกับเอกสารของชุดข้อมูลก่อนเผยแพร่ผล และเปรียบเทียบอัตราการสมัครตามจำนวนครั้ง

In [ ]:
if "campaign" in df.columns:
    display(run_sql("""
    SELECT
        campaign AS contact_attempts_in_campaign,
        COUNT(*) AS records,
        SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) AS subscriptions,
        ROUND(100.0 * SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS subscription_rate_pct
    FROM bank_marketing
    GROUP BY campaign
    ORDER BY campaign;
    """))
else:
    print("ไม่พบคอลัมน์ campaign ในชุดข้อมูลนี้")


### แบบฝึก SQL 6 — ตรวจสัดส่วนค่า `unknown`

ใช้ SQL ตรวจค่าหมวดหมู่ `unknown` ในคอลัมน์ที่พบบ่อย หากไม่มีคอลัมน์ใดมีค่าแบบนี้ ตารางจะไม่แสดงแถว

In [ ]:
unknown_sql_parts = []
for col in categorical_cols:
    # Column names are sourced from the actual CSV; quote identifiers safely for SQLite.
    quoted = '"' + col.replace('"', '""') + '"'
    unknown_sql_parts.append(f"""
    SELECT '{col.replace("'", "''")}' AS column_name,
           SUM(CASE WHEN LOWER(TRIM(CAST({quoted} AS TEXT))) = 'unknown' THEN 1 ELSE 0 END) AS unknown_count,
           ROUND(100.0 * SUM(CASE WHEN LOWER(TRIM(CAST({quoted} AS TEXT))) = 'unknown' THEN 1 ELSE 0 END) / COUNT(*), 2) AS unknown_pct
    FROM bank_marketing
    """)
if unknown_sql_parts:
    display(run_sql(" UNION ALL ".join(unknown_sql_parts) + " ORDER BY unknown_pct DESC"))
else:
    print("ไม่มีคอลัมน์หมวดหมู่ให้ตรวจสอบ")


## 7. แบบฝึกสรุปความหมายทางธุรกิจ (Business Insight)

ใช้ผลลัพธ์จริงที่ได้จากการรันโค้ดเพื่อตอบคำถามต่อไปนี้

1. **กลุ่มที่มีอัตราสมัครสูง:** กลุ่มใดมีอัตราการสมัครสูงที่สุด และมีจำนวนลูกค้ากี่ราย?
2. **อัตราเทียบกับปริมาณ:** กลุ่มที่มีอัตราสูงมีจำนวนลูกค้ามากพอที่จะนำไปทดลองทำแคมเปญหรือไม่?
3. **ข้อเสนอแนะทางธุรกิจ:** ธนาคารควรทดลองแคมเปญขนาดเล็กแบบใดที่สามารถวัดผลได้?
4. **ข้อจำกัด:** มีปัจจัยอื่นใดที่อาจอธิบายความแตกต่างระหว่างกลุ่มได้?
5. **ตัวชี้วัด:** จะประเมินการทดลองอย่างไร เช่น อัตราสมัคร จำนวนผู้สมัคร ต้นทุนต่อผู้สมัคร และจำนวนลูกค้าที่ติดต่อ?

อย่าสรุปว่ากลุ่มใด “ทำให้” ลูกค้าสมัครมากขึ้นจากผลเชิงพรรณนาเพียงอย่างเดียว ควรเขียนว่า “กลุ่มนี้มีความสัมพันธ์กับอัตราการสมัครที่สูงกว่าในชุดข้อมูลนี้”

## 8. ส่งออกรายงานคุณภาพข้อมูล (ไม่บังคับ)
Cell นี้จะสร้างไฟล์ CSV สรุปชนิดข้อมูล จำนวนค่าว่าง และสัดส่วนค่าว่างของแต่ละคอลัมน์

In [ ]:
quality_report = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(dtype) for dtype in df.dtypes],
    "missing_count": [int(df[col].isna().sum()) for col in df.columns],
    "missing_pct": [round(float(df[col].isna().mean() * 100), 2) for col in df.columns],
})
quality_report.to_csv("data_quality_report.csv", index=False)
print("Saved data_quality_report.csv")


## 9. สรุปก่อนเผยแพร่

- เขียน Insight จากค่าที่ได้หลังรัน Notebook กับไฟล์ `train.csv` จริงเท่านั้น
- ระบุทั้งอัตราการสมัครและจำนวนระเบียนของแต่ละกลุ่ม
- รายงานการจัดการค่า `unknown` และค่าว่างอย่างโปร่งใส
- ตรวจสอบผลลัพธ์ทั้งหมดก่อนนำตัวเลขไปใส่ README หรือ GitHub
- ดูไฟล์ `README.md` ที่มาพร้อมกับโปรเจกต์สำหรับวิธีใช้งาน ข้อจำกัด และแนวทางการตีความ

แหล่งข้อมูล: [Kaggle Playground Series S5E8](https://www.kaggle.com/competitions/playground-series-s5e8/data)